# MedGraphRAG Projesi: Veri Hazırlama ve Ön İşleme

Bu not defteri, **MedGraphRAG** projesi kapsamında tıbbi literatür (Gastrointestinal Sistem ve Otoimmünite bölümleri) PDF'lerinin işlenmesi, ayrıştırılması ve veri gürültüsünün temizlenmesi adımlarını içermektedir.

Büyük Dil Modelleri (LLM) ve Graph-RAG sistemlerinin halüsinasyon görmesini engellemek için, metinlerin referans, sayfa numarası ve alt bilgi/üst bilgi gibi gürültülerden arındırılması kritik bir öneme sahiptir.



In [1]:
!pip install -q haystack-ai pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 709.5/709.5 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.8/240.8 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 2.7 MB/s eta 0:00:00


## Adım 1: Temel PDF Okuma (PyPDF ve Haystack Pipeline)

İlk olarak Haystack ekosistemi ve basit `PyPDFToDocument` bileşenini kullanarak PDF belgelerimizi okuyup anlamlı kelime öbeklerine böleceğimiz bir hat (pipeline) kuruyoruz.

*Not: Bu yöntem hızlıdır ancak çıktı metinlerinde sayfa kalıntıları ve akademik gürültüler (header/footer vb.) bırakabilir. Kaliteyi görmek için bir test yapıyoruz.*

In [2]:
import os
from haystack.components.converters import PyPDFToDocument
from haystack.components.preprocessors import DocumentSplitter
from haystack import Pipeline

In [ ]:
# 1. Klasör yolunu belirleyelim (Colab'a yüklediğiniz Chapter 23 klasörü)
pdf_dir = "/content/drive/MyDrive/MedGraphRAG/Resources/chapter 23"

# Klasördeki PDF dosyalarını bulalım
pdf_files = [os.path.join(pdf_dir, f) for f in os.listdir(pdf_dir) if f.endswith(".pdf")]
print(f"Test için {len(pdf_files)} adet PDF bulundu.")

# 2. Haystack Bileşenleri
# Yapay zeka tabanlı gelişmiş temizlik için PyPDFToDocument veya Unstructured bileşeni kullanıyoruz
converter = PyPDFToDocument()
# Anlam bütünlüğü için kelime bazlı 300 kelimelik ve 50 kelime örtüşmeli (overlap) parçalar
splitter = DocumentSplitter(split_by="word", split_length=300, split_overlap=50)

# 3. Pipeline Kurulumu
indexing_pipeline = Pipeline()
indexing_pipeline.add_component("converter", converter)
indexing_pipeline.add_component("splitter", splitter)
indexing_pipeline.connect("converter", "splitter")

# 4. Pipeline'ı Çalıştırma ve Test
if len(pdf_files) > 0:
    print("Dokümanlar işleniyor, lütfen bekleyin...")
    results = indexing_pipeline.run({"converter": {"sources": pdf_files}})
    documents = results["splitter"]["documents"]

    print(f"\n✅ İşlem Tamamlandı! Toplam elde edilen temiz doküman parçası (chunk) sayısı: {len(documents)}")

    # Metin temizliğinin (gürültü, tablo, referans) kalitesini gözlemlemek için ilk 2 chunk'ı yazdıralım
    print("\n--- ÖRNEK METİN PARÇASI 1 ---")
    print(documents[0].content)

    print("\n--- ÖRNEK METİN PARÇASI 2 ---")
    print(documents[1].content)
else:
    print("Klasörde PDF bulunamadı. Lütfen sol taraftaki panelden dosyaların doğru klasöre yüklendiğinden emin olun.")

Test için 39 adet PDF bulundu.
Dokümanlar işleniyor, lütfen bekleyin...

✅ İşlem Tamamlandı! Toplam elde edilen temiz doküman parçası (chunk) sayısı: 1493

--- ÖRNEK METİN PARÇASI 1 ---
Allergy and the gastrointestinal system
G. Vighi,* F. Marcucci,‡ L. Sensi,‡
G. Di Cara‡ and F. Frati‡
*Clinical Pharmacology and Pharmacovigilance,
Niguarda Ca’ Granda Hospital, Milan, and
‡Institute of Pediatrics, Department of Medical
and Surgical Specialties and Public Health,
Perugia, Italy
Summary
The gastrointestinal system plays a central role in immune system homeo-
stasis. It is the main route of contact with the external environment and is
overloaded every day with external stimuli, sometimes dangerous as patho-
gens (bacteria, protozoa, fungi, viruses) or toxic substances, in other cases
very useful as food or commensal ﬂora. The crucial position of the gas-
trointestinal system is testiﬁed by the huge amount of immune cells that
reside within it. Indeed, gut-associated lymphoid tissue (GAL T

## Adım 2: Gelişmiş PDF Temizleme (Unstructured ile)

İlk adımdaki çıktılarda, metinlerin içinde anlamsız akademik tablo kalıntıları, telif hakları yazıları ve sayfa numaralarının kaldığını gözlemledik.

Bilgi Çizgesi (Knowledge Graph) oluştururken modelimizin (örn. Gemma 4 / 7B) yanlış varlıklar (entity) ve ilişkiler çıkarmasını önlemek amacıyla **Unstructured** kütüphanesine geçiş yapıyoruz. Bu araç metinlerin yapısal analizini (Header, Title, Text) ayırarak gürültüleri bizim için filtreleyecek.

In [ ]:
# API yönlendirmesi yapan paketi siliyoruz
!pip uninstall -y unstructured-fileconverter-haystack

# Colab sistemine yerel PDF işleme motorlarını kuruyoruz
!apt-get update -qq
!apt-get install -y -qq poppler-utils tesseract-ocr

# Unstructured kütüphanesini yerel yetenekleriyle kuruyoruz
!pip install -q "unstructured[pdf]" haystack-ai

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package poppler-utils.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing triggers for man-db (2.10.2-1) ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 26.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 8.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━

In [ ]:
from unstructured.partition.pdf import partition_pdf
from haystack import Document
from haystack.components.preprocessors import DocumentSplitter

## Adım 3: Gürültüsüz Veri Hattını Çalıştırma (Chunking)

Unstructured kütüphanesinin `fast` stratejisini kullanarak metinleri çekiyoruz. Sadece anlamlı Ana Metin ve Başlıkları (Text, NarrativeText, Title) filtreleyip, Haystack yapısına uygun `Document` formatına dönüştüreceğiz. Ardından DocumentSplitter ile örtüşmeli parçalara (chunk) böleceğiz.

In [ ]:
# Klasör yolunuz
pdf_dir = "/content/drive/MyDrive/MedGraphRAG/Resources/chapter 23"
pdf_files = [os.path.join(pdf_dir, f) for f in os.listdir(pdf_dir) if f.endswith(".pdf")]

print(f"\nİşlenecek PDF sayısı: {len(pdf_files)}")

# 2. API Olmadan Yerel (Local) Unstructured Kullanımı
raw_documents = []
if len(pdf_files) > 0:
    print("Yerel ortamda (API'siz) yapay zeka destekli metin temizliği yapılıyor...")
    print("Bu işlem Colab işlemcilerini kullanacağı için biraz sürebilir, lütfen bekleyin.\n")

    for file_path in pdf_files:
        try:
            # 'fast' stratejisi metinleri hızlıca çeker.
            # Header, Footer, Title, Text gibi yapısal analizleri kendi içinde yapar.
            elements = partition_pdf(filename=file_path, strategy="fast")

            # Sadece anlamlı metin bloklarını alıp alt/üst bilgi (Header/Footer) gürültülerini filtreliyoruz
            clean_text_blocks = []
            for el in elements:
                # Sadece Ana Metin (Text, NarrativeText) ve Başlıkları (Title) alıyoruz
                if el.category in ["NarrativeText", "Text", "Title"]:
                    clean_text_blocks.append(str(el))

            full_clean_text = "\n\n".join(clean_text_blocks)

            # Haystack'in anlayacağı Document formatına çeviriyoruz
            raw_documents.append(Document(content=full_clean_text, meta={"file_name": os.path.basename(file_path)}))
            print(f"✅ İşlendi: {os.path.basename(file_path)}")

        except Exception as e:
            print(f"❌ Hata ({os.path.basename(file_path)}): {e}")

# 3. Haystack DocumentSplitter ile parçalama (Chunking)
if len(raw_documents) > 0:
    splitter = DocumentSplitter(split_by="word", split_length=300, split_overlap=50)

    # Yerel olarak temizlediğimiz belgeleri Haystack splitter'ına veriyoruz
    splitted_docs = splitter.run(documents=raw_documents)["documents"]

    print(f"\n✅ Tüm işlemler tamam! Toplam elde edilen pürüzsüz doküman parçası (chunk) sayısı: {len(splitted_docs)}")

    print("\n--- TEMİZLENMİŞ ÖRNEK METİN PARÇASI ---")
    print(splitted_docs[0].content)
else:
    print("\n❌ Klasörde PDF bulunamadı veya metin çıkarılamadı.")


İşlenecek PDF sayısı: 39
Yerel ortamda (API'siz) yapay zeka destekli metin temizliği yapılıyor...
Bu işlem Colab işlemcilerini kullanacağı için biraz sürebilir, lütfen bekleyin.



✅ İşlendi: Allergy and the gastrointestinal system.pdf


✅ İşlendi: The evolution of the danger theory.pdf


✅ İşlendi: ageing-and-the-gut.pdf


✅ İşlendi: A natural therapeutic approach for the treatment of periodontitis.pdf


...

✅ İşlendi: Saccharomyces boulardii CNCM I-745 supports regeneration of the intestinal microbiota after diarrheic dysbiosis   a review.pdf
✅ İşlendi: wunna-HOW THE IMMUNE SYSTEM WORKS FIFTH EDITION BY LAUREN SOMPAYRAC-685af7a31b37e1.72352688.pdf


✅ İşlendi: Shaping the oral microbiota through intimate kissing.pdf
✅ İşlendi: Exposure to household furry pets influences the gut microbiota of infants at 3–4 months following various birth scenarios.pdf


✅ İşlendi: Having older siblings is associated with gut microbiota development during early childhood.pdf


✅ İşlendi: Early-Life Events, Including Mode of Delivery and Type of Feeding, Siblings and Gender, Shape the Developing Gut Microbiota.pdf


✅ İşlendi: Maternal modifiers of the infant gut microbiota metabolic consequences.pdf

✅ Tüm işlemler tamam! Toplam elde edilen pürüzsüz doküman parçası (chunk) sayısı: 771

--- TEMİZLENMİŞ ÖRNEK METİN PARÇASI ---
Allergy and the gastrointestinal system

Perugia, Italy

Accepted for publication April 23 2008

Correspondence: F. Marcucci, Institute of Pedi-

atrics, Department of Medical and Surgical

Specialties and Public Health, Policlinico

Summary

The gastrointestinal system plays a central role in immune system homeo- stasis. It is the main route of contact with the external environment and is overloaded every day with external stimuli, sometimes dangerous as patho- gens (bacteria, protozoa, fungi, viruses) or toxic substances, in other cases very useful as food or commensal ﬂora. The crucial position of the gas- trointestinal system is testiﬁed by the huge amount of immune cells that reside within it. Indeed, gut-associated lymphoid tissue (GALT) is the prominent part of mucosal

## Adım 4: Neo4j Graf Veritabanı Bağlantısı

Çıkaracağımız karmaşık tıbbi ilişkileri standart tablolarda değil, birbirine bağlarla bağlı "Düğümler" (Nodes) ve "İlişkiler" (Edges) şeklinde tutacağız. Bunun için bulut tabanlı **Neo4j AuraDB** kullanıyoruz.

In [3]:
# Neo4j Python sürücüsünü kuralım
!pip install -q neo4j

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 12.8 MB/s eta 0:00:00


In [4]:
from neo4j import GraphDatabase

In [12]:
# Neo4j AuraDB'den aldığınız bilgileri buraya girin
NEO4J_URI = ".."
NEO4J_USERNAME = ".."
NEO4J_PASSWORD = ".."

def test_connection(uri, user, password):
    try:
        # Veritabanına bağlanmayı deniyoruz
        driver = GraphDatabase.driver(uri, auth=(user, password))
        driver.verify_connectivity()
        print("✅ Neo4j AuraDB Bağlantısı Başarıyla Kuruldu! Graph veritabanımız veri almaya hazır.")
        return driver
    except Exception as e:
        print(f"❌ Bağlantı Hatası: {e}")
        print("Lütfen URI ve Şifrenizi doğru kopyaladığınızdan emin olun.")
        return None

# Bağlantıyı test et
driver = test_connection(NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD)

✅ Neo4j AuraDB Bağlantısı Başarıyla Kuruldu! Graph veritabanımız veri almaya hazır.


## Adım 5: LLM (Gemma 7B) Hazırlığı ve 4-Bit Kuantalama

Gemma 7B gibi 7 milyar parametreli devasa bir modeli Colab'ın sunduğu ücretsiz 16 GB bellekli (T4) GPU'ya sığdırmak için **4-bit Kuantalama (Quantization)** kullanıyoruz. Bu işlem, modelin ağırlıklarını sıkıştırarak bellek kullanımını dramatik şekilde düşürür.

In [6]:
# GPU üzerinde 4-bit optimizasyon ve model indirme için gerekli paketler
!pip install -q accelerate bitsandbytes huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.7 MB/s eta 0:00:00


In [7]:
import torch
from haystack.components.generators import HuggingFaceLocalGenerator
from haystack.components.builders import PromptBuilder
from transformers import BitsAndBytesConfig

In [8]:
# 1. HuggingFace Token'ını Doğrudan Ortama Atama
os.environ["HF_TOKEN"] = ".."

In [9]:
# 2. GÜNCEL 4-Bit Kuantalama Ayarları (BitsAndBytesConfig ile)
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

generator = HuggingFaceLocalGenerator(
    model="google/gemma-1.1-2b-it",
    task="text-generation",
    generation_kwargs={"max_new_tokens": 400, "temperature": 0.1},
    huggingface_pipeline_kwargs={
        "model_kwargs": {
            "quantization_config": quantization_config
        }
    }
)

## Adım 6: Varlık ve İlişki Çıkarım Hattı (Extraction Pipeline)

LLM'e özel bir komut (Prompt) vererek, ona verdiğimiz her tıbbi paragraftan ilgili tıbbi varlıkları ve nedensellik ilişkilerini standart bir **JSON** formatında çıkarmasını söyleyeceğiz.

In [ ]:
# 3. Varlık ve İlişki Çıkarımı İçin Özel Prompt Şablonu
prompt_template = """
Sen uzman bir tıbbi bilgi çıkarım (Knowledge Graph) asistanısın.
Aşağıdaki tıbbi metinden hastalıklar (Disease), semptomlar (Symptom), anatomik yapılar (Anatomy), bağışıklık sistemi bileşenleri (Immune_Component) ve tedaviler (Treatment) gibi temel varlıkları çıkar.
Ayrıca bu varlıklar arasındaki ilişkileri (örneğin; "CAUSES", "TREATS", "LOCATED_IN", "PREVENTS") belirle.

Metin:
{{document_content}}

Lütfen SADECE aşağıdaki JSON formatında çıktı ver. JSON dışında hiçbir ek metin, açıklama veya markdown satırı (```json gibi) KULLANMA:
{
  "entities": [
    {"id": "varlık_adı", "type": "Varlık_Tipi"}
  ],
  "relations": [
    {"source": "kaynak_varlık", "relation": "İLİŞKİ_TİPİ", "target": "hedef_varlık"}
  ]
}
"""

prompt_builder = PromptBuilder(template=prompt_template)

# 4. Haystack Pipeline Kurulumu
kg_extraction_pipeline = Pipeline()
kg_extraction_pipeline.add_component("prompt_builder", prompt_builder)
kg_extraction_pipeline.add_component("llm", generator)
kg_extraction_pipeline.connect("prompt_builder", "llm")

# 5. İlk Doküman Üzerinde Test
if 'splitted_docs' in locals() and len(splitted_docs) > 0:
    test_doc = splitted_docs[0].content

    print("Gemma modeli belleğe yükleniyor ve varlık çıkarımı yapılıyor...")
    print("(Model zaten indirildiği için bu sefer sadece belleğe yüklenme süresi alacaktır)")

    result = kg_extraction_pipeline.run({
        "prompt_builder": {"document_content": test_doc}
    })

    # Modelin ürettiği JSON çıktısını alalım
    llm_output = result["llm"]["replies"][0]

    print("\n✅ İŞLEM TAMAMLANDI! GEMMA'NIN ÇIKARDIĞI JSON VERİSİ:\n")
    print(llm_output)
else:
    print("❌ Hata: Önceki adımdan kalan 'splitted_docs' listesi bulunamadı. Lütfen parçalama (chunking) kodunu çalıştırdığınızdan emin olun.")

Gemma modeli belleğe yükleniyor ve varlık çıkarımı yapılıyor...
(Model zaten indirildiği için bu sefer sadece belleğe yüklenme süresi alacaktır)


Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



✅ İŞLEM TAMAMLANDI! GEMMA'NIN ÇIKARDIĞI JSON VERİSİ:


```

**Varlıklar (Disease)**

- Allergy
- Gastrointestinal system

**Sinptomlar (Symptom)**

- Allerji
- Gastrointestinal sistem sorunları

**Anatomik yapılar (Anatomy)**

- Gastrointestinal sistem

**Bağışıklık sistemi bileşenleri (Immune_Component)**

- Innate bağışıklık sistem
- Adaptive bağışıklık sistem

**Tedaviler (Treatment)**

- Nörolojik tedavi
- İmmün tedaviler
- Biyolojik tedavi


 **!!!** Burada model .json verisi çıkartamadı. Bunun için aşağıdaki yeni kod bloğumuz ile GPU önbelleği boşaltıldı ve modelden tekrar .json verisi çıkartması istendi.

In [ ]:
import gc

In [ ]:
# 1. ESKİ MODELİ EKRAN KARTINDAN (GPU) TEMİZLEME
if 'kg_extraction_pipeline' in locals():
    del kg_extraction_pipeline
if 'generator' in locals():
    del generator

# Çöp toplayıcıyı çalıştır ve PyTorch GPU önbelleğini (cache) boşalt
gc.collect()
torch.cuda.empty_cache()
print("✅ GPU belleği eski modelden başarıyla temizlendi!\n")

# 2. YENİ 7B MODELİNİ GÜVENLİ YÜKLEME
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

# GÜNCELLEME: device_map="auto" eklendi. Bu ayar, modelin ağırlıklarını
# GPU'daki boşluklara en verimli şekilde dağıtarak bellek taşmasını önler.
generator = HuggingFaceLocalGenerator(
    model="google/gemma-1.1-7b-it",
    task="text-generation",
    generation_kwargs={"max_new_tokens": 500, "do_sample": False},
    huggingface_pipeline_kwargs={
        "device_map": "auto",
        "model_kwargs": {
            "quantization_config": quantization_config
        }
    }
)

# 3. PROMPT VE PIPELINE KURULUMU
prompt_template = """
Sen uzman bir tıbbi bilgi çıkarım (Knowledge Graph) asistanısın.
Aşağıdaki tıbbi metinden hastalıklar (Disease), semptomlar (Symptom), anatomik yapılar (Anatomy), bağışıklık sistemi bileşenleri (Immune_Component) ve tedaviler (Treatment) gibi temel varlıkları çıkar.
Ayrıca bu varlıklar arasındaki ilişkileri (örneğin; "CAUSES", "TREATS", "LOCATED_IN", "PREVENTS") belirle.

Metin:
{{document_content}}

Lütfen SADECE aşağıdaki JSON formatında çıktı ver. JSON dışında hiçbir ek metin, açıklama veya markdown satırı (```json gibi) KULLANMA:
{
  "entities": [
    {"id": "varlık_adı", "type": "Varlık_Tipi"}
  ],
  "relations": [
    {"source": "kaynak_varlık", "relation": "İLİŞKİ_TİPİ", "target": "hedef_varlık"}
  ]
}
"""

prompt_builder = PromptBuilder(template=prompt_template)

kg_extraction_pipeline = Pipeline()
kg_extraction_pipeline.add_component("prompt_builder", prompt_builder)
kg_extraction_pipeline.add_component("llm", generator)
kg_extraction_pipeline.connect("prompt_builder", "llm")

# 4. TEST
if 'splitted_docs' in locals() and len(splitted_docs) > 0:
    test_doc = splitted_docs[0].content

    print("Gemma 7B modeli belleğe yükleniyor ve varlık çıkarımı yapılıyor...")

    result = kg_extraction_pipeline.run({
        "prompt_builder": {"document_content": test_doc}
    })

    llm_output = result["llm"]["replies"][0]

    print("\n🎉 İŞLEM TAMAMLANDI! GEMMA 7B'NİN ÇIKARDIĞI JSON VERİSİ:\n")
    print(llm_output)
else:
    print("❌ Hata: 'splitted_docs' bulunamadı.")

✅ GPU belleği eski modelden başarıyla temizlendi!

Gemma 7B modeli belleğe yükleniyor ve varlık çıkarımı yapılıyor...


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🎉 İŞLEM TAMAMLANDI! GEMMA 7B'NİN ÇIKARDIĞI JSON VERİSİ:


```

**Sonuç:**
```json
{
  "entities": [
    {"id": "Gastrointestinal_System", "type": "Anatomy"},
    {"id": "Immune_System", "type": "Immune_Component"},
    {"id": "Gut-Associated_Lymph_Tissue", "type": "Anatomy"},
    {"id": "Plasma_Cells", "type": "Immune_Component"},
    {"id": "IgA", "type": "Immune_Component"},
    {"id": "Allergy", "type": "Disease"},
    {"id": "Coeliac_Disease", "type": "Disease"},
    {"id": "Food_Allergy", "type": "Disease"},
    {"id": "Antigen-Presenting_Cells", "type": "Immune_Component"}
  ],
  "relations": [
    {"source": "Gastrointestinal_System", "relation": "LOCATED_IN", "target": "Immune_System"},
    {"source": "Gastrointestinal_System", "relation": "OVERLOADS", "target": "External_Environment"},
    {"source": "Gut-Associated_Lymph_Tissue", "relation": "REPRESENTS", "target": "Immune_System"},
    {"source": "Gut-Associated_Lymph_Tissue", "relation": "INTERACTS", "target": "Gastrointes

## Adım 7: Çıktıları Ayrıştırma ve Neo4j'ye Kaydetme

Modelden aldığımız JSON verisini ayıklayıp doğrudan Neo4j veritabanına `MERGE` (varsa güncelle, yoksa oluştur) Cypher komutlarıyla yazacak Python fonksiyonumuzu oluşturuyoruz.

In [10]:
import json
import re

In [ ]:
# LLM çıktısının içinden sadece JSON kısmını (süslü parantezler arasını) Regex ile bulalım
# Bu yöntem, model başa veya sona ne kadar saçma metin eklerse eklesin JSON'ı bulur.
match = re.search(r'\{.*\}', llm_output, re.DOTALL)

if match:
    clean_json_string = match.group(0)
    try:
        kg_data = json.loads(clean_json_string)
        print("✅ JSON verisi başarıyla ayrıştırıldı ve çözümlendi!")
    except json.JSONDecodeError as e:
        print(f"❌ JSON format hatası: {e}")
        kg_data = None
else:
    print("❌ Hata: Metin içinde geçerli bir JSON bloğu (süslü parantez) bulunamadı.")
    kg_data = None

# Neo4j'ye Veri Yazan Fonksiyonumuz
def write_to_neo4j(tx, data):
    # 1. Düğümleri (Nodes) Veritabanına Ekle
    entities = data.get("entities", [])
    for entity in entities:
        node_id = entity["id"]
        node_type = entity["type"].replace(" ", "_") # Etiketlerde boşluk olmaması için

        # MERGE komutu, düğüm (node) zaten varsa üzerine yazar, yoksa yeni oluşturur
        query = f"MERGE (n:`{node_type}` {{id: $id}})"
        tx.run(query, id=node_id)

    print(f"✅ {len(entities)} adet düğüm (node) veritabanına aktarıldı.")

    # 2. İlişkileri (Relations) Veritabanına Ekle
    relations = data.get("relations", [])
    for rel in relations:
        source = rel["source"]
        target = rel["target"]
        rel_type = rel["relation"].replace(" ", "_").upper() # Neo4j standartları gereği BÜYÜK HARF

        query = f"""
        MATCH (a {{id: $source}})
        MATCH (b {{id: $target}})
        MERGE (a)-[r:`{rel_type}`]->(b)
        """
        tx.run(query, source=source, target=target)

    print(f"✅ {len(relations)} adet ilişki (relation) veritabanına aktarıldı.")

# Fonksiyonu çalıştırarak verileri Neo4j AuraDB'ye gönderelim
if kg_data and 'driver' in locals():
    with driver.session() as session:
        session.execute_write(write_to_neo4j, kg_data)
        print("\n🎉 TEBRİKLER! İlk metin parçanız başarıyla Neo4j Bilgi Çizgesi'ne (Knowledge Graph) eklendi!")
elif 'driver' not in locals():
    print("❌ Hata: Neo4j 'driver' nesnesi bulunamadı. Lütfen veritabanı bağlantı kodunu çalıştırdığınızdan emin olun.")

✅ JSON verisi başarıyla ayrıştırıldı ve çözümlendi!
✅ 9 adet düğüm (node) veritabanına aktarıldı.
✅ 10 adet ilişki (relation) veritabanına aktarıldı.

🎉 TEBRİKLER! İlk metin parçanız başarıyla Neo4j Bilgi Çizgesi'ne (Knowledge Graph) eklendi!


Knowledge Graph'ımızı drive'a kaydediyoruz.

In [ ]:
# 1. Drive üzerinde çıktıları kaydedeceğimiz klasörü oluşturalım
output_dir = "/content/drive/MyDrive/MedGraphRAG/Outputs"
os.makedirs(output_dir, exist_ok=True)

# 2. Haystack Document objelerini JSON formatına dönüştürelim
chunks_data = []
if 'splitted_docs' in locals():
    for i, doc in enumerate(splitted_docs):
        chunks_data.append({
            "chunk_id": i,
            "content": doc.content,
            "meta": doc.meta # Dosya adı gibi meta verileri içerir
        })

    # 3. Drive'a kaydetme
    chunks_file_path = os.path.join(output_dir, "cleaned_chunks.json")
    with open(chunks_file_path, "w", encoding="utf-8") as f:
        json.dump(chunks_data, f, ensure_ascii=False, indent=4)

    print(f"✅ Başarılı! Toplam {len(chunks_data)} adet temizlenmiş metin parçası (chunk) şu konuma kaydedildi:\n{chunks_file_path}")
else:
    print("❌ Hata: 'splitted_docs' listesi bulunamadı. Lütfen veri hazırlama adımını çalıştırdığınızdan emin olun.")

✅ Başarılı! Toplam 771 adet temizlenmiş metin parçası (chunk) şu konuma kaydedildi:
/content/drive/MyDrive/MedGraphRAG/Outputs/cleaned_chunks.json


## Adım 8: Toplu İşlem (Batch Processing) ve Drive'a Yedekleme

Son adım olarak elimizdeki yüzlerce temiz metin parçasının tümünü döngü içerisinde LLM'e gönderecek, cevaplarını işleyecek ve işlem yarıda kesilirse diye verileri Google Drive'a yedekleyeceğiz.

In [32]:
from tqdm.notebook import tqdm

In [ ]:
output_dir = "/content/drive/MyDrive/MedGraphRAG/Outputs"
os.makedirs(output_dir, exist_ok=True)
graphs_file_path = os.path.join(output_dir, "extracted_graphs.json")

# 1. Daha Önce Kaydedilmiş Verileri Yükle (Kaldığı Yerden Devam Etmek İçin)
if os.path.exists(graphs_file_path):
    with open(graphs_file_path, "r", encoding="utf-8") as f:
        all_extracted_data = json.load(f)
    print(f"📂 Drive'da {len(all_extracted_data)} adet işlenmiş veri bulundu. Kaldığı yerden devam edilecek...")
else:
    all_extracted_data = []
    print("📂 Yeni bir kayıt dosyası oluşturuluyor...")

# Sadece henüz işlenmemiş olan metin parçalarını alıyoruz
docs_to_process = splitted_docs[len(all_extracted_data):]

print(f"\n⏳ İşlenecek kalan doküman parçası sayısı: {len(docs_to_process)}\n")

basarili_chunk = 0
hatali_chunk = 0

# 2. Toplu İşlem ve Kaydetme Döngüsü
for i, doc in enumerate(tqdm(docs_to_process, desc="Knowledge Graph Örülüyor")):
    try:
        # Gemma 7B'den Varlık ve İlişki Çıkarımı İste
        result = kg_extraction_pipeline.run({
            "prompt_builder": {"document_content": doc.content}
        })
        llm_output = result["llm"]["replies"][0]

        # Regex ile sadece JSON kısmını ayıkla
        match = re.search(r'\{.*\}', llm_output, re.DOTALL)

        if not match:
            hatali_chunk += 1
            continue

        clean_json_string = match.group(0)
        kg_data = json.loads(clean_json_string)

        # Meta veriyi de JSON'a ekleyelim (Hangi kaynaktan geldiğini bilmek için)
        kg_data["source_file"] = doc.meta.get("file_name", "unknown")

        # Neo4j'ye Yazdır
        with driver.session() as session:
            # write_to_neo4j fonksiyonunun içerisindeki printleri yoruma alabilirsiniz (ekranı doldurmaması için)
            session.execute_write(write_to_neo4j, kg_data)

        # Başarılı veriyi ana listemize ekle
        all_extracted_data.append(kg_data)

        # HER BAŞARILI ADIMDA DOSYAYI GÜNCELLE (Checkpoint)
        with open(graphs_file_path, "w", encoding="utf-8") as f:
            json.dump(all_extracted_data, f, ensure_ascii=False, indent=4)

        basarili_chunk += 1

    except Exception as e:
        hatali_chunk += 1
        pass

print(f"\n🎉 DÖNGÜ TAMAMLANDI VEYA DURDURULDU!")
print(f"✅ Bu oturumda başarıyla işlenen ve Drive'a kaydedilen parça: {basarili_chunk}")
print(f"❌ Format/Çıkarım Hatası Sebebiyle Atlanan Sayı: {hatali_chunk}")
print(f"📁 Tüm verilerinizin güvende olduğu dosya: {graphs_file_path}")

📂 Yeni bir kayıt dosyası oluşturuluyor...

⏳ İşlenecek kalan doküman parçası sayısı: 771



Knowledge Graph Örülüyor:   0%|          | 0/771 [00:00<?, ?it/s]

Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ 9 adet düğüm (node) veritabanına aktarıldı.


Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

✅ 10 adet ilişki (relation) veritabanına aktarıldı.
✅ 5 adet düğüm (node) veritabanına aktarıldı.


Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ 8 adet ilişki (relation) veritabanına aktarıldı.
✅ 10 adet düğüm (node) veritabanına aktarıldı.


Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ 9 adet düğüm (node) veritabanına aktarıldı.
✅ 8 adet ilişki (relation) veritabanına aktarıldı.

🎉 DÖNGÜ TAMAMLANDI VEYA DURDURULDU!
✅ Bu oturumda başarıyla işlenen ve Drive'a kaydedilen parça: 547
❌ Format/Çıkarım Hatası Sebebiyle Atlanan Sayı: 224
📁 Tüm verilerinizin güvende olduğu dosya: /content/drive/MyDrive/MedGraphRAG/Outputs/extracted_graphs.json


## Adım 9: Boru Hattını (Pipeline) Manuel Orkestre Etme ve Test

Verilerimizi Neo4j'ye kaydettik ve 7 Milyar parametreli Gemma modelimizi 4-Bit formatında belleğe yükledik. Artık sistemimize tıbbi sorular sorarak, oluşturduğumuz bilgi çizgesinden (Knowledge Graph) faydalanmasını sağlayabiliriz.

Büyük Dil Modellerini (LLM) Colab'ın sınırlı GPU (T4) ortamında çalıştırırken, standart aracı kütüphanelerin (Haystack, LangChain vb.) otomatik `Pipeline` zincirleri bazen karmaşıklık yaratıp bellek tahsisi hatalarına (CUDA Out of Memory) yol açabilmektedir.

Bu sorunu aşmak (workaround) ve hata ayıklamayı (debugging) kolaylaştırmak için burada harika bir mühendislik yaklaşımı izliyoruz: **Bileşenleri sırayla ve manuel olarak (Sequential Execution) çalıştırıyoruz.**

**Çalışma Mantığı:**
1. **Retriever (Geri Çağırıcı):** Sorumuzdaki kelimeleri alır, Neo4j'ye gider ve `A --[CAUSES]--> B` şeklindeki tıbbi ilişkileri çeker.
2. **Prompt Builder (Komut Üretici):** Çekilen bu graf verilerini ve kullanıcının sorusunu birleştirip LLM'e gidecek nihai talimatı hazırlar.
3. **Generator (Üretici):** Gemma 7B modeli bu promptu okur, graf üzerinde mantıksal nedensellik kurar ve uydurmadan (halüsinasyonsuz) cevap üretir.

In [13]:
from haystack import component

In [24]:
# 1. HAYSTACK İÇİN ÖZEL NEO4J GRAPH RETRIEVER BİLEŞENİ
@component
class Neo4jGraphRetriever:
    def __init__(self, uri, user, password):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))
        # Gereksiz bağlaçları (stop words) filtreleyelim ki ağda gürültü aramayalım
        self.stopwords = ["nedir", "nelerdir", "nasıl", "hangi", "neden", "niçin", "için", "göre",
                          "olarak", "olan", "veya", "gibi", "arasındaki", "bir", "bu", "ile", "kadar"]

    @component.output_types(graph_context=str)
    def run(self, query: str):
        # Sorudaki kelimeleri temizleyip anahtar kelimeleri çıkarıyoruz
        words = query.lower().replace("?", "").replace(".", "").replace(",", "").split()
        keywords = [w for w in words if len(w) > 3 and w not in self.stopwords]

        context_relations = []

        # Neo4j üzerinde Cypher sorgusu
        with self.driver.session() as session:
            for kw in keywords:
                # Anahtar kelimeyi içeren düğümleri (Node) ve bağlı oldukları ilişkileri (Relation) bul
                cypher_query = """
                MATCH (a)-[r]->(b)
                WHERE toLower(a.id) CONTAINS $kw OR toLower(b.id) CONTAINS $kw
                RETURN a.id AS source, type(r) AS relation, b.id AS target LIMIT 15
                """
                result = session.run(cypher_query, kw=kw)

                for record in result:
                    # İlişkiyi okunabilir bir metne dönüştür: "GALT --[REPRESENTS]--> Immune_System"
                    rel_str = f"{record['source']} --[{record['relation']}]--> {record['target']}"
                    if rel_str not in context_relations:
                        context_relations.append(rel_str)

        if not context_relations:
            return {"graph_context": "Bilgi çizgesinde (Graph) bu soruyla ilgili doğrudan bir ilişki bulunamadı."}

        # Tüm ilişkileri alt alta bir metin bloğu haline getiriyoruz
        return {"graph_context": "\n".join(context_relations)}

graph_retriever = Neo4jGraphRetriever(NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD)

# 3. GRAPH-RAG İÇİN ZIRHLI (ANTI-HALÜSİNASYON) PROMPT ŞABLONU
graph_prompt_template = """
Sen uzman, son derece dikkatli ve analitik düşünen bir immünoloji profesörüsün.
Aşağıda, tıbbi literatürden çıkarılmış "Bilgi Çizgesi (Knowledge Graph)" ilişkileri verilmiştir. (Kaynak Varlık --[İLİŞKİ_TİPİ]--> Hedef Varlık).

KESİN KURALLARIN:
1. SADECE ama SADECE aşağıdaki graf ilişkilerini (Context) kullanacaksın. Kendi genel tıbbi bilgini veya internetten öğrendiklerini KESİNLİKLE KULLANMA.
2. Soruda geçen spesifik maddeler (örneğin ilaç isimleri, genler) graf verisinde YOKSA, tıbbi bir sonuç uydurma (halüsinasyon yapma).
3. Akıl Yürütme (Reasoning) Adımları:
   - Önce graf verisinde sorudaki kavramlarla ilgili neler olduğunu açıkla.
   - Ardından sorudaki spesifik kavramların grafta olup olmadığını kontrol et.
   - Eğer sorunun kesin cevabı grafta yoksa, cevabını mutlaka şu cümleyle bitir: "Ancak graf verilerinde bu spesifik maddeler/ilişkiler bulunmadığı için kesin bir sonuca varılamamaktadır."

Graf İlişkileri (Context):
{{graph_context}}

Soru: {{question}}
Adım Adım Akıl Yürütme ve Cevap:
"""

graph_prompt_builder = PromptBuilder(template=graph_prompt_template)

In [22]:
test_sorusu = "Gastrointestinal sistemin bağışıklık sistemindeki rolü nedir ve GALT'ın (gut-associated lymphoid tissue) buradaki yüzdesel payı nedir?"
ideal_cevap = "Bilinmiyor (Manuel test)"

print(f"🎯 TEST EDİLEN SORU: {test_sorusu}\n")
print("Neo4j Bilgi Çizgesi taranıyor...\n")

# HATA DÜZELTİLDİ: Pipeline kullanmadan bileşenleri sırayla doğrudan çalıştırıyoruz!

# 1. Adım: Neo4j'den Graf Verisini (Context) Çek
retriever_result = graph_retriever.run(query=test_sorusu)
graf_baglami = retriever_result["graph_context"]

# 2. Adım: Prompt'u Oluştur
prompt_result = graph_prompt_builder.run(graph_context=graf_baglami, question=test_sorusu)
hazir_prompt = prompt_result["prompt"]

print("Gemma 7B graf (ilişki) verisiyle yanıt üretiyor, lütfen bekleyin...\n")

# 3. Adım: Hazır Prompt'u doğrudan Gemma 7B'ye (generator) ver
response_graph = generator.run(prompt=hazir_prompt)
graf_cevabi = response_graph["replies"][0]

print("==================================================")
print("🧠 GRAPH-RAG SİSTEMİNİN CEVABI (Neo4j Destekli):")
print("==================================================")
print(graf_cevabi)
print("\n")
print("==================================================")
print("👨‍⚕️ İDEAL UZMAN CEVABI (Altın Standart):")
print("==================================================")
print(ideal_cevap)

# Arka planda tam olarak hangi okların (ilişkilerin) bulunduğunu görelim
print("\n🔍 Arka Planda Neo4j'den Çekilen Graf İlişkileri (Context):")
print(graf_baglami)

🎯 TEST EDİLEN SORU: Gastrointestinal sistemin bağışıklık sistemindeki rolü nedir ve GALT'ın (gut-associated lymphoid tissue) buradaki yüzdesel payı nedir?

Neo4j Bilgi Çizgesi taranıyor...



Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Gemma 7B graf (ilişki) verisiyle yanıt üretiyor, lütfen bekleyin...

🧠 GRAPH-RAG SİSTEMİNİN CEVABI (Neo4j Destekli):


Gastrointestinal sistemin bağışıklık sistemindeki rolü, hastalığın teşhisini koyma ve tedavi edilmesini destekler. GALT'ın (gut-associated lymphoid tissue) yüzdesel payı, bağışıklık sisteminin fonksiyonlarını belirleyen önemli bir rol oynar.


👨‍⚕️ İDEAL UZMAN CEVABI (Altın Standart):
Bilinmiyor (Manuel test)

🔍 Arka Planda Neo4j'den Çekilen Graf İlişkileri (Context):
Gastrointestinal_System --[LOCATED_IN]--> Immune_System
Allergy --[CAUSES]--> Gastrointestinal_System
Coeliac_Disease --[CAUSES]--> Gastrointestinal_System
Food_Allergy --[CAUSES]--> Gastrointestinal_System
Treatment --[TREATS]--> Gastrointestinal_Anaphylaxis
Gastrointestinal_Anaphylaxis --[CAUSES]--> Immune_Complex
Cow's_Milk --[CAUSES]--> Gastrointestinal_Anaphylaxis
Hen's_Egg --[CAUSES]--> Gastrointestinal_Anaphylaxis
Intestinal_Level --[LOCATED_IN]--> Gastrointestinal_Anaphylaxis
Symptoms --[DESCRIBES

In [25]:
test_sorusu = "Immune suppression by lysosomotropic amines and cyclosporine on T-cell responses to minor and major histocompatibility antigens: does synergy exist?"
ideal_cevap = "Lysosomotropic amines in combination with cyclosporine appear to be synergistic in the suppression of T-cell proliferation to MiHC and MHC. Use of chloroquine in combination with cyclosporine may result in improved control of GVHD."

print(f"🎯 TEST EDİLEN SORU: {test_sorusu}\n")
print("Neo4j Bilgi Çizgesi taranıyor...\n")

# HATA DÜZELTİLDİ: Pipeline kullanmadan bileşenleri sırayla doğrudan çalıştırıyoruz!

# 1. Adım: Neo4j'den Graf Verisini (Context) Çek
retriever_result = graph_retriever.run(query=test_sorusu)
graf_baglami = retriever_result["graph_context"]

# 2. Adım: Prompt'u Oluştur
prompt_result = graph_prompt_builder.run(graph_context=graf_baglami, question=test_sorusu)
hazir_prompt = prompt_result["prompt"]

print("Gemma 7B graf (ilişki) verisiyle yanıt üretiyor, lütfen bekleyin...\n")

# 3. Adım: Hazır Prompt'u doğrudan Gemma 7B'ye (generator) ver
response_graph = generator.run(prompt=hazir_prompt)
graf_cevabi = response_graph["replies"][0]

print("==================================================")
print("🧠 GRAPH-RAG SİSTEMİNİN CEVABI (Neo4j Destekli):")
print("==================================================")
print(graf_cevabi)
print("\n")
print("==================================================")
print("👨‍⚕️ İDEAL UZMAN CEVABI (Altın Standart):")
print("==================================================")
print(ideal_cevap)

# Arka planda tam olarak hangi okların (ilişkilerin) bulunduğunu görelim
print("\n🔍 Arka Planda Neo4j'den Çekilen Graf İlişkileri (Context):")
print(graf_baglami)

🎯 TEST EDİLEN SORU: Immune suppression by lysosomotropic amines and cyclosporine on T-cell responses to minor and major histocompatibility antigens: does synergy exist?

Neo4j Bilgi Çizgesi taranıyor...



Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Gemma 7B graf (ilişki) verisiyle yanıt üretiyor, lütfen bekleyin...

🧠 GRAPH-RAG SİSTEMİNİN CEVABI (Neo4j Destekli):


1. **Soru kapsamı:** Bu graf ilişkileri, hastalığın immün sisteminin nasıl etkileşimli olduğunu gösterir.
2. **Graf ilişkileri:**
   - Immune_System ile Tissue arasında yerleşir.
   - Immune_System, Asthma ve Cancer gibi hastalıkların önlenmesi için yardımcı olabilir.
   - T- hücrelerin immün sistem tarafından CONSUMEDILİyor ve DAMAGEDILİyor olabilir.
3. **Cevap:** Ancak graf verilerinde bu spesifik maddeler/ilişkiler bulunmadığı için kesin bir sonuca varılamamaktadır.


👨‍⚕️ İDEAL UZMAN CEVABI (Altın Standart):
Lysosomotropic amines in combination with cyclosporine appear to be synergistic in the suppression of T-cell proliferation to MiHC and MHC. Use of chloroquine in combination with cyclosporine may result in improved control of GVHD.

🔍 Arka Planda Neo4j'den Çekilen Graf İlişkileri (Context):
Gastrointestinal_System --[LOCATED_IN]--> Immune_System
Immune_System -

# Adım 10: Sentetik Veri ile A/B Test

In [34]:
import pandas as pd

In [30]:
# 1. 5 Soruluk Altın Standart Veri Setimiz
golden_qa_dataset = [
    {
        "id": 1,
        "question": "Otoimmün hastalıkların gelişiminde bağırsak bariyerinin (intestinal barrier) bozulması nasıl bir rol oynar ve bu durum bağışıklık sistemini nasıl tetikler?",
        "ideal_answer": "Bağırsak bariyerinin bozulması (sızıntılı bağırsak/leaky gut), sindirilmemiş besin proteinlerinin, toksinlerin ve patojenlerin kan dolaşımına geçmesine neden olur. GALT (Gut-Associated Lymphoid Tissue) bu yabancı maddeleri tehdit olarak algılar ve aşırı bir bağışıklık tepkisi başlatır. Bu kronik inflamasyon, moleküler taklit (molecular mimicry) gibi mekanizmalarla bağışıklık sisteminin kendi dokularına saldırmasına (otoimmünite) yol açar."
    },
    {
        "id": 2,
        "question": "Gastrointestinal sistemin bağışıklık sistemindeki yeri nedir ve GALT (Gut-Associated Lymphoid Tissue) buradaki savunma mekanizmasına yüzdesel olarak ne kadar katkı sağlar?",
        "ideal_answer": "Gastrointestinal sistem, vücudun en büyük bağışıklık organıdır. GALT (Bağırsak İlişkili Lenfoid Doku), tüm bağışıklık sisteminin yaklaşık %70 ila %80'ini oluşturur ve dış dünyadan gelen patojenlere karşı ilk ve en büyük savunma hattını temsil eder."
    },
    {
        "id": 3,
        "question": "Besin alerjilerinde (Food Allergy) plazma hücreleri tarafından üretilen IgA antikorlarının mukozal yüzeylerdeki temel görevi nedir?",
        "ideal_answer": "Plazma hücreleri tarafından üretilen salgısal IgA antikorları, mukozal yüzeylerde bir 'bağışıklık boyası' veya bariyer olarak işlev görür. Besin antijenlerine veya patojenlere bağlanarak onların bağırsak epitelinden kana sızmasını (translokasyon) engeller ve böylece alerjik reaksiyonların ve inflamasyonun önüne geçer (Immune Exclusion)."
    },
    {
        "id": 4,
        "question": "Çölyak hastalığı (Coeliac Disease) gibi durumlarda, antijen sunan hücrelerin (Antigen-Presenting Cells) bağırsak mukozasındaki işlevi normalden nasıl sapar?",
        "ideal_answer": "Normalde antijen sunan hücreler (APC'ler), bağırsak mukozasında tolerans (hoşgörü) geliştirmekle görevlidir. Ancak Çölyak hastalığı gibi durumlarda bariyer bozulduğunda, APC'ler gluten gibi proteinleri tehlikeli patojenler olarak algılar ve T-hücrelerine sunarak doku hasarına yol açan şiddetli bir inflamatuar kaskad (nedensellik zinciri) başlatır."
    },
    {
        "id": 5,
        "question": "Mikrobiyotanın dengesinin bozulması (Disbiyoz), gastrointestinal fonksiyonları bozarak sistemik enflamasyon ve nörolojik semptomlara (Gut-Brain Axis) nasıl yol açabilir?",
        "ideal_answer": "Disbiyoz, bağırsaktaki faydalı bakterilerin azalıp zararlı bakterilerin artmasıdır. Bu durum kısa zincirli yağ asitlerinin (SCFA) üretimini azaltır, bağırsak bariyerini zayıflatır (sızıntı) ve pro-enflamatuar sitokinlerin kana karışmasına neden olur. Bu sitokinler kan-beyin bariyerini aşarak vagus siniri üzerinden nörolojik semptomlara ve sistemik enflamasyona yol açar."
    }
]

In [35]:
print(f"🧬 Graph-RAG Testi Başlıyor! {len(golden_qa_dataset)} Altın Standart Soru sorulacak...\n")

graph_rag_sonuclari = []

for item in tqdm(golden_qa_dataset, desc="Graph-RAG Soruları Cevaplıyor"):
    soru = item["question"]
    ideal_cevap = item["ideal_answer"]

    try:
        # 1. Adım: Neo4j'den Graf Verisini (Context) Çek
        retriever_result = graph_retriever.run(query=soru)
        graf_baglami = retriever_result["graph_context"]

        # 2. Adım: Anti-Halüsinasyon Prompt'u Oluştur
        prompt_result = graph_prompt_builder.run(graph_context=graf_baglami, question=soru)
        hazir_prompt = prompt_result["prompt"]

        # 3. Adım: Gemma 7B'ye (generator) Gönder
        response_graph = generator.run(prompt=hazir_prompt)
        graf_cevabi = response_graph["replies"][0].strip()

    except Exception as e:
        graf_cevabi = f"Hata: {e}"
        graf_baglami = "Alınamadı"

    # Sonuçları listeye kaydet
    graph_rag_sonuclari.append({
        "Soru": soru,
        "İdeal Uzman Cevabı": ideal_cevap,
        "Model B (Graph-RAG)": graf_cevabi,
        "Graph Tarafından Bulunan İlişkiler (Bağlam)": graf_baglami
    })

# DataFrame oluştur ve Excel'e kaydet
df_graph_sonuclar = pd.DataFrame(graph_rag_sonuclari)

output_dir = "/content/drive/MyDrive/MedGraphRAG/Outputs"
# Eğer klasör yoksa oluştur
os.makedirs(output_dir, exist_ok=True)

excel_path = os.path.join(output_dir, "MedGraphRAG_Graph_Test_Raporu.xlsx")

# Excel olarak kaydet
df_graph_sonuclar.to_excel(excel_path, index=False)

print("\n🎉 GRAPH-RAG TESTİ TAMAMLANDI!")
print(f"📊 Sadece Graph-RAG sonuçlarını içeren rapor başarıyla oluşturuldu: {excel_path}")

🧬 Graph-RAG Testi Başlıyor! 5 Altın Standart Soru sorulacak...



Graph-RAG Soruları Cevaplıyor:   0%|          | 0/5 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentati


🎉 GRAPH-RAG TESTİ TAMAMLANDI!
📊 Sadece Graph-RAG sonuçlarını içeren rapor başarıyla oluşturuldu: /content/drive/MyDrive/MedGraphRAG/Outputs/MedGraphRAG_Graph_Test_Raporu.xlsx
